In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon
from matplotlib.ticker import FuncFormatter

def mb_formatter(x, pos):  # Mb ticks
    return f"{x/1e6:.1f} Mb"

def add_block(ax, ref_start, ref_end, qry_start=None, qry_end=None,
              facecolor="#4C78A8", edgecolor="none", alpha=0.6, hatch=None):
    """
    Draw a filled synteny block connecting:
      - Reference interval on the y-axis: [ref_start, ref_end] at x=0
      - Query interval on the x-axis: [qry_start, qry_end] at y=0
    If qry_start==qry_end (or qry_end is None), the block tapers to a point.
    """
    if qry_end is None:  # allow single x position
        qry_end = qry_start

    # Clamp order
    y0, y1 = sorted([ref_start, ref_end])
    x0, x1 = sorted([qry_start, qry_end])

    # Polygon with vertices running around the perimeter:
    # left edge on y-axis, bottom edge on x-axis
    # If x0==x1, this becomes a triangle.
    verts = [(0, y0), (0, y1), (x1, 0), (x0, 0)]
    poly = Polygon(verts, closed=True, facecolor=facecolor, edgecolor=edgecolor,
                   alpha=alpha, hatch=hatch)
    ax.add_patch(poly)

def plot_synteny_blocks(events, x_label="Query genome (x)", y_label="Reference genome (y)",
                        fig_size=(7,7)):
    """
    events: list of dicts with keys:
      type: 'DEL'|'DUP'|'INS'|'INV' (affects styling only)
      ref_start, ref_end: coordinates on reference (bp)
      qry_start, qry_end: coordinates on query (bp); for DEL, set qry_start=ref_start and omit/==qry_end
                          for DUP, qry interval wider than ref
                          for INS, set ref_start≈ref_end (point) and give qry interval
    """
    fig, ax = plt.subplots(figsize=fig_size)

    for ev in events:
        t = ev["type"].upper()
        ref_start, ref_end = ev["ref_start"], ev["ref_end"]
        qry_start, qry_end = ev.get("qry_start"), ev.get("qry_end")

        # Styling by SV type
        if t == "DEL":
            color, hatch = "#D64D4D", None          # triangle to a point on x
            if qry_end is None:                     # allow shorthand
                qry_end = qry_start
        elif t == "DUP":
            color, hatch = "#2E7EBB", None          # trapezoid widening on x
        elif t == "INS":
            color, hatch = "#5AA469", None          # trapezoid widening on x from a point on y
        elif t == "INV":
            color, hatch = "#A06CD5", "//"          # same geometry; hatch indicates inversion
        else:
            color, hatch = "#7F7F7F", None

        add_block(ax, ref_start, ref_end, qry_start, qry_end, facecolor=color, hatch=hatch)

    # Axes & formatting
    ax.set_xlabel(x_label)
    ax.set_ylabel(y_label)
    ax.xaxis.set_major_formatter(FuncFormatter(mb_formatter))
    ax.yaxis.set_major_formatter(FuncFormatter(mb_formatter))
    ax.grid(True, alpha=0.3, linewidth=0.6)

    # Nice bounds
    # Expand to include all polygons comfortably
    xs = []
    ys = []
    for p in ax.patches:
        vx, vy = zip(*p.get_xy())
        xs.extend(vx); ys.extend(vy)
    pad_x = max(1, (max(xs) - min(xs)) * 0.05) if xs else 1
    pad_y = max(1, (max(ys) - min(ys)) * 0.05) if ys else 1
    ax.set_xlim(min(0, min(xs, default=0)) - pad_x, max(xs, default=1) + pad_x)
    ax.set_ylim(min(0, min(ys, default=0)) - pad_y, max(ys, default=1) + pad_y)
    ax.set_aspect("equal", adjustable="box")
    plt.tight_layout()
    return fig, ax

# ---- Examples ----
# 1) Deletion on chr6 between 50–75 (bp shown for simplicity; use real bp in practice):
example_events = [
    # DEL: reference y spans 50–75, query collapses to a point at x=50
    {"type": "DEL", "ref_start": 50, "ref_end": 75, "qry_start": 50},

    # DUP: reference 100–130 maps to a wider interval on x (e.g., 100–160)
    {"type": "DUP", "ref_start": 100, "ref_end": 130, "qry_start": 100, "qry_end": 160},

    # INS: small insertion at ref position 200, expands to 200–215 on x
    {"type": "INS", "ref_start": 200, "ref_end": 200, "qry_start": 200, "qry_end": 215},

    # INV: inverted mapping (geometry same; style via hatch)
    {"type": "INV", "ref_start": 260, "ref_end": 300, "qry_start": 300, "qry_end": 260},
]

fig, ax = plot_synteny_blocks(example_events)
plt.show()
